<a href="https://colab.research.google.com/github/GAYATHRI-100/HACKTHON-INNOMATICS/blob/main/Fine_tune_a_transformer_model_(BERT_DistilBERT)_to_perform_Part_of_Speech_(POS)_Tagging_Chunking_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("juliangarratt/conll2003-dataset")

print("Path to dataset files:", path)

100%|██████████| 960k/960k [00:00<00:00, 118MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/juliangarratt/conll2003-dataset/versions/1


In [3]:
!pip install transformers datasets seqeval evaluate -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.7 MB/s eta 0:00:00


In [4]:
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer
import evaluate

In [6]:
import os

data_path = "/root/.cache/kagglehub/datasets/juliangarratt/conll2003-dataset/versions/1"

print(os.listdir(data_path))

['conll2003']


In [7]:
data_path = "/root/.cache/kagglehub/datasets/juliangarratt/conll2003-dataset/versions/1/conll2003"

import os
print(os.listdir(data_path))

['eng.testa', 'eng.train', 'eng.testb']


In [8]:
train_path = f"{data_path}/eng.train"
val_path = f"{data_path}/eng.testa"
test_path = f"{data_path}/eng.testb"

In [9]:
def read_conll(file_path):
    tokens = []
    pos_tags = []
    chunk_tags = []

    all_tokens = []
    all_pos = []
    all_chunks = []

    with open(file_path, "r") as f:
        for line in f:
            if line.strip() == "" or line.startswith("-DOCSTART-"):
                if tokens:
                    all_tokens.append(tokens)
                    all_pos.append(pos_tags)
                    all_chunks.append(chunk_tags)
                    tokens, pos_tags, chunk_tags = [], [], []
            else:
                parts = line.strip().split()

                # Ensure correct parsing
                if len(parts) >= 3:
                    tokens.append(parts[0])
                    pos_tags.append(parts[1])
                    chunk_tags.append(parts[2])  # 👈 use this for chunking

    return {
        "tokens": all_tokens,
        "pos_tags": all_pos,
        "chunk_tags": all_chunks
    }

In [10]:
train_data = read_conll(train_path)
val_data = read_conll(val_path)
test_data = read_conll(test_path)

In [11]:
print(train_data["tokens"][0])
print(train_data["chunk_tags"][0])
print(len(train_data["tokens"][0]), len(train_data["chunk_tags"][0]))

['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.']
['B-NP', 'B-VP', 'B-NP', 'I-NP', 'B-VP', 'I-VP', 'B-NP', 'I-NP', 'O']
9 9


In [12]:
from datasets import Dataset, DatasetDict

dataset = DatasetDict({
    "train": Dataset.from_dict(train_data),
    "validation": Dataset.from_dict(val_data),
    "test": Dataset.from_dict(test_data),
})

In [13]:
chunk_labels = list(set(tag for doc in train_data["chunk_tags"] for tag in doc))
chunk_labels.sort()

label2id = {label: i for i, label in enumerate(chunk_labels)}
id2label = {i: label for label, i in label2id.items()}

In [14]:
chunk_labels = list(set(tag for doc in train_data["chunk_tags"] for tag in doc))
chunk_labels.sort()

label2id = {label: i for i, label in enumerate(chunk_labels)}
id2label = {i: label for label, i in label2id.items()}

In [16]:
!pip install transformers datasets seqeval evaluate -q

In [17]:
import os
import numpy as np
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer
import evaluate

In [18]:
data_path = "/root/.cache/kagglehub/datasets/juliangarratt/conll2003-dataset/versions/1/conll2003"

train_path = f"{data_path}/eng.train"
val_path = f"{data_path}/eng.testa"
test_path = f"{data_path}/eng.testb"

In [19]:
def read_conll(file_path):
    tokens, pos_tags, chunk_tags = [], [], []
    all_tokens, all_pos, all_chunks = [], [], []

    with open(file_path, "r") as f:
        for line in f:
            if line.strip() == "" or line.startswith("-DOCSTART-"):
                if tokens:
                    all_tokens.append(tokens)
                    all_pos.append(pos_tags)
                    all_chunks.append(chunk_tags)
                    tokens, pos_tags, chunk_tags = [], [], []
            else:
                parts = line.strip().split()
                if len(parts) >= 3:
                    tokens.append(parts[0])
                    pos_tags.append(parts[1])
                    chunk_tags.append(parts[2])

    return {
        "tokens": all_tokens,
        "pos_tags": all_pos,
        "chunk_tags": all_chunks
    }

In [20]:
train_data = read_conll(train_path)
val_data = read_conll(val_path)
test_data = read_conll(test_path)

In [21]:
dataset = DatasetDict({
    "train": Dataset.from_dict(train_data),
    "validation": Dataset.from_dict(val_data),
    "test": Dataset.from_dict(test_data),
})

In [22]:
all_tags = []

for split in [train_data, val_data, test_data]:
    for doc in split["chunk_tags"]:
        all_tags.extend(doc)

chunk_labels = sorted(list(set(all_tags)))

label2id = {label: i for i, label in enumerate(chunk_labels)}
id2label = {i: label for label, i in label2id.items()}

print(chunk_labels)

['B-ADJP', 'B-ADVP', 'B-CONJP', 'B-INTJ', 'B-LST', 'B-NP', 'B-PP', 'B-PRT', 'B-SBAR', 'B-VP', 'I-ADJP', 'I-ADVP', 'I-CONJP', 'I-INTJ', 'I-LST', 'I-NP', 'I-PP', 'I-PRT', 'I-SBAR', 'I-VP', 'O']


In [23]:
def encode_labels(example):
    example["labels"] = [label2id[tag] for tag in example["chunk_tags"]]
    return example

dataset = dataset.map(encode_labels)

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

In [24]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []

    for i, label in enumerate(examples["labels"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(label[word_idx])

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [25]:
tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

In [26]:
model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(chunk_labels),
    id2label=id2label,
    label2id=label2id
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [28]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=1,
    logging_steps=50
)

In [30]:
import numpy as np
import evaluate

metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [chunk_labels[p] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    true_labels = [
        [chunk_labels[l] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
    }

In [36]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)

In [37]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"].select(range(2000)),
    eval_dataset=tokenized_datasets["validation"].select(range(500)),
    compute_metrics=compute_metrics,
    data_collator=data_collator,
)

In [38]:
trainer.train()
trainer.evaluate()

Step,Training Loss
50,1.934056
100,1.007835
150,0.694666
200,0.565551
250,0.442133
300,0.453862
350,0.484022
400,0.445382
450,0.357271
500,0.478086


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


{'eval_loss': 0.4020879566669464,
 'eval_precision': 0.8420595892392247,
 'eval_recall': 0.8239456552504953,
 'eval_f1': 0.8329041487839771,
 'eval_runtime': 16.894,
 'eval_samples_per_second': 29.596,
 'eval_steps_per_second': 7.399,
 'epoch': 1.0}

In [39]:
def predict(sentence):
    tokens = sentence.split()
    inputs = tokenizer(tokens, is_split_into_words=True, return_tensors="pt", truncation=True)

    outputs = model(**inputs)
    predictions = outputs.logits.argmax(dim=2)

    word_ids = inputs.word_ids()
    predicted_labels = []
    previous_word_idx = None

    for idx, word_idx in enumerate(word_ids):
        if word_idx is None:
            continue
        elif word_idx != previous_word_idx:
            predicted_labels.append(chunk_labels[predictions[0][idx]])
        previous_word_idx = word_idx

    return list(zip(tokens, predicted_labels))

In [40]:
print(predict("John works at Google in California"))

[('John', 'B-NP'), ('works', 'B-VP'), ('at', 'B-PP'), ('Google', 'B-NP'), ('in', 'B-PP'), ('California', 'B-NP')]


In [41]:
# =========================
# 1. Install Libraries
# =========================
!pip install transformers datasets seqeval -q

# =========================
# 2. Imports
# =========================
import os
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForTokenClassification, Trainer, TrainingArguments
import numpy as np
from seqeval.metrics import classification_report, f1_score

# =========================
# 3. Dataset Path
# =========================
data_path = "/root/.cache/kagglehub/datasets/juliangarratt/conll2003-dataset/versions/1/conll2003"

train_file = os.path.join(data_path, "eng.train")
val_file   = os.path.join(data_path, "eng.testa")
test_file  = os.path.join(data_path, "eng.testb")

# =========================
# 4. Read CoNLL File
# =========================
def read_conll(file_path):
    sentences = []
    tokens = []
    pos_tags = []

    with open(file_path, "r") as f:
        for line in f:
            if line.strip() == "":
                if tokens:
                    sentences.append({"tokens": tokens, "pos_tags": pos_tags})
                    tokens, pos_tags = [], []
            else:
                parts = line.strip().split()
                tokens.append(parts[0])
                pos_tags.append(parts[1])

    return sentences

train_data = read_conll(train_file)
val_data   = read_conll(val_file)
test_data  = read_conll(test_file)

# =========================
# 5. Create Dataset
# =========================
dataset = DatasetDict({
    "train": Dataset.from_list(train_data),
    "validation": Dataset.from_list(val_data),
    "test": Dataset.from_list(test_data),
})

# =========================
# 6. Label Mapping
# =========================
label_list = sorted({tag for d in train_data for tag in d["pos_tags"]})
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}

def encode_labels(example):
    example["labels"] = [label2id[tag] for tag in example["pos_tags"]]
    return example

dataset = dataset.map(encode_labels)

# =========================
# 7. Tokenizer
# =========================
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        padding="max_length",
        max_length=128,
        is_split_into_words=True
    )

    labels = []
    for i, label in enumerate(examples["labels"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []
        prev_word = None

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != prev_word:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            prev_word = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)

# =========================
# 8. Model
# =========================
model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

# =========================
# 9. Metrics
# =========================
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_labels = []
    true_preds = []

    for pred, lab in zip(predictions, labels):
        curr_labels = []
        curr_preds = []
        for p_i, l_i in zip(pred, lab):
            if l_i != -100:
                curr_labels.append(id2label[l_i])
                curr_preds.append(id2label[p_i])
        true_labels.append(curr_labels)
        true_preds.append(curr_preds)

    return {"f1": f1_score(true_labels, true_preds)}

# =========================
# 10. Training Arguments
# =========================
training_args = TrainingArguments(
    output_dir="./pos_results",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=1,
    logging_steps=50,
    save_steps=200,
    do_eval=True
)

# =========================
# 11. Trainer (NO tokenizer param)
# =========================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"].select(range(2000)),
    eval_dataset=tokenized_datasets["validation"].select(range(500)),
    compute_metrics=compute_metrics
)

# =========================
# 12. Train
# =========================
trainer.train()

# =========================
# 13. Evaluate
# =========================
results = trainer.evaluate()
print(results)

Map:   0%|          | 0/14987 [00:00<?, ? examples/s]

Map:   0%|          | 0/3466 [00:00<?, ? examples/s]

Map:   0%|          | 0/3684 [00:00<?, ? examples/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': 0.5177006721496582, 'eval_f1': 0.8415412676698586, 'eval_runtime': 142.6746, 'eval_samples_per_second': 3.504, 'eval_steps_per_second': 0.876, 'epoch': 1.0}


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: -X- seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NNP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: : seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: IN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWar

In [42]:
def predict(sentence):
    tokens = sentence.split()
    inputs = tokenizer(tokens, return_tensors="pt", is_split_into_words=True)

    outputs = model(**inputs).logits
    preds = np.argmax(outputs.detach().numpy(), axis=2)

    word_ids = inputs.word_ids()
    final = []

    for i, word_id in enumerate(word_ids):
        if word_id is not None:
            final.append((tokens[word_id], id2label[preds[0][i]]))

    return final

print(predict("John works at Google in California"))

[('John', 'NNP'), ('works', 'VBZ'), ('at', 'IN'), ('Google', 'NNP'), ('in', 'IN'), ('California', 'NNP')]


Task 7 Compare:
POS Tagging → Grammar-level tagging
Chunking → Phrase-level grouping


| Aspect                    | POS Tagging                           | Chunking                                     |
| ------------------------- | ------------------------------------- | -------------------------------------------- |
| Core Idea                 | Assigns grammatical role to each word | Groups words into meaningful phrases         |
| Level                     | Word-level                            | Phrase-level                                 |
| Output Type               | Single tag per word                   | Sequence of phrase boundary tags (B-, I-, O) |
| Example                   | John/NN works/VB                      | John (B-NP), works (B-VP)                    |
| Context Dependency        | Low                                   | High                                         |
| Complexity                | Easy                                  | Medium                                       |
| Model Requirement         | Basic sequence labeling               | Context-aware sequence modeling              |
| Use of Syntax             | Limited                               | Uses shallow syntax (phrase structure)       |
| Error Sensitivity         | Lower                                 | Higher (wrong boundary affects phrase)       |
| Applications              | Grammar checking, text analysis       | Information extraction, NER, parsing         |
| Learning Difficulty       | Easier to train                       | Harder due to dependencies between words     |
| Label Types               | NN, VB, JJ, etc.                      | B-NP, I-NP, B-VP, O, etc.                    |
| Dependency Between Tokens | Independent                           | Dependent (phrase continuity matters)        |
| Evaluation Complexity     | Simple                                | More complex (phrase correctness matters)    |


POS tagging treats each word independently, while chunking depends on surrounding words to form meaningful phrases.
Chunking uses BIO tagging scheme (Begin, Inside, Outside), which makes it more structured than POS tagging.
Errors in POS tagging usually affect only one word, but errors in chunking can affect entire phrases.
Chunking is a step closer to syntactic parsing, whereas POS tagging is more basic.
POS tagging is often used as a preprocessing step for chunking.
Chunking requires better context understanding, which is why transformer models like BERT perform well.

Task 8: Report

Write about:
Differences between POS Tagging and Chunking


Part-of-Speech (POS) tagging and chunking are both important tasks in Natural Language Processing, but they operate at different levels of linguistic analysis.

POS tagging is the process of assigning grammatical labels to each word in a sentence, such as noun, verb, adjective, etc. It works at the word level, meaning each word is treated independently and tagged based on its role in the sentence. For example, in the sentence “John works at Google”, “John” is tagged as a noun and “works” as a verb.

Chunking, also known as shallow parsing, goes a step further by grouping words into meaningful phrases such as noun phrases (NP), verb phrases (VP), and prepositional phrases (PP). It works at the phrase level and considers the relationship between words. For example, “John” may be labeled as part of a noun phrase (B-NP), while “works” is part of a verb phrase (B-VP).

The key difference lies in their level of analysis: POS tagging focuses on identifying the grammatical category of individual words, whereas chunking focuses on identifying the structure of phrases within a sentence. As a result, chunking is more complex because it requires understanding the context and boundaries between words.
Additionally, POS tagging typically uses simple labels like NN (noun) or VB (verb), while chunking uses the BIO tagging scheme (Begin, Inside, Outside) to define phrase boundaries. Errors in POS tagging affect only individual words, whereas errors in chunking can impact entire phrases.

In summary, POS tagging is a foundational task that provides basic grammatical information, while chunking builds on this information to identify higher-level phrase structures, making it more context-dependent and slightly more challenging.

Challenges faced


During the implementation of POS tagging and chunking using transformer models, several challenges are:

Dataset Format Handling:
The CoNLL-2003 dataset is in a custom text format, not a standard CSV/JSON. Parsing sentences and separating tokens, POS tags, and chunk tags correctly required careful preprocessing.
Label Inconsistency Across Splits:
Some chunk labels (e.g., I-PRT) appeared only in validation/test sets and not in the training set, which caused KeyError. This was resolved by creating label mappings using all dataset splits.
Tokenization and Subword Alignment:
BERT tokenization splits words into subwords, making it difficult to align original labels with tokenized outputs. Proper handling using word_ids() and assigning -100 to special tokens was necessary.
Handling Special Tokens:
Tokens like [CLS] and [SEP] do not have labels, so assigning -100 was required to ignore them during loss computation.
Batch Padding Issues:
Variable-length sequences caused errors during training. This was resolved by using a DataCollator to dynamically pad sequences within each batch.
Library Version Compatibility:
Differences in transformers versions caused errors (e.g., unsupported arguments like evaluation_strategy and tokenizer). Adjustments were needed to match the environment.
Memory Constraints (RAM Issues):
Training on the full dataset caused crashes. This was handled by reducing dataset size, batch size, and number of epochs.
Model Training Stability:
With limited epochs, some rare labels were not predicted, leading to warnings in evaluation metrics. Increasing training data or epochs can improve performance.

**Observations and insights**


During the implementation of POS tagging and chunking using transformer-based models, several important observations were made:

POS tagging achieved stable performance quickly because it is a simpler, word-level task. The model was able to correctly identify most grammatical categories such as nouns, verbs, and prepositions even with limited training data.
Chunking was comparatively more complex, as it required understanding the relationship between words to form meaningful phrases. This made the task more dependent on context and slightly harder to learn.
The use of transformer models like BERT significantly improved performance for both tasks, especially in capturing contextual information compared to traditional models.
Tokenization played a critical role, particularly due to subword splitting in BERT. Proper alignment of labels with tokens was essential to avoid incorrect predictions.
It was observed that most errors in chunking occurred at phrase boundaries, where the model struggled to correctly assign B- (Begin) and I- (Inside) tags.
In POS tagging, errors were less impactful, as they affected only individual words, whereas in chunking, a single mistake could affect the entire phrase structure.
Evaluation metrics showed that the model achieved good precision and recall, but some rare labels had low scores due to insufficient training examples.
Reducing dataset size (due to memory constraints) slightly impacted performance, but the model still generalized well for common patterns.